## © 2026 Noob Dev — Confidential Course Material

This project is part of the **Certified AI Engineer Professional Program** conducted by **Noob Dev**.

---

### ⚠️ Strict Confidentiality & Academic Integrity Notice

All project files — including this notebook, any source code, the knowledge base, the evaluation dataset, and any related materials — are the **exclusive intellectual property of Noob Dev** and are provided **solely for the personal use of enrolled program participants**.

**You may NOT:**

- Share, copy, redistribute, or publish any part of this project in any form
- Upload it to public repositories (GitHub, GitLab, Hugging Face, Kaggle, etc.) or to AI training datasets
- Forward it to non-enrolled individuals, study groups outside the cohort, or online communities
- Use it for commercial purposes outside the program
- Submit it as your own work to any other course, bootcamp, employer, or platform

**Violations constitute academic misconduct and a breach of your program enrollment agreement. Consequences include:**

- Immediate **termination from the Certified AI Engineer Professional Program**
- Forfeiture of all tuition fees, course progress, certifications, and program credentials
- **Legal action** under applicable copyright, trade-secret, and intellectual property laws

By opening, downloading, or using these materials, you acknowledge and accept these terms.

---


# Project 02 — Project Requirements & Guidance

## RAG-Powered Support Assistant for TechNova

This notebook is your **roadmap** for the project. It contains:
- The learning objectives
- A breakdown of every required component
- Step-by-step guidance for how to approach each step
- Hints, questions to think about, and common mistakes to avoid

> **⚠️ Important:** This notebook contains **NO code on purpose**. You are expected to write all the code yourself in a separate notebook called `solution.ipynb`. Use this notebook as a guide, not a template.

---
## 1. Learning Objectives

By the end of this project, you should be able to:

1. **Ingest** real-world documents from a folder structure into a usable format for RAG.
2. **Chunk** documents intelligently using size, overlap, and metadata.
3. **Embed** text into vectors using a model of your choice and persist them in a vector store.
4. **Retrieve** the most relevant chunks for a given query, including with metadata filtering.
5. **Generate** grounded answers using an LLM and properly cite sources.
6. **Evaluate** a RAG system using a structured test set.
7. **Deploy** a working chat UI for end-users.

These are the same building blocks used in real-world enterprise RAG systems. Master these and you can build a production knowledge assistant for any company.

---
## 2. Project Overview

You will build a **chatbot for the internal support team at TechNova**. The team gets dozens of repetitive questions every day about products, returns, warranties, shipping, and account issues. They need an assistant that answers fast, accurately, and with **sources** — so support reps can verify what the bot says before quoting it to a real customer.

**Why this matters:** A real customer-support RAG system has zero tolerance for hallucinations. If the bot makes up a return policy, the company is legally liable. This is why **source citation** and **honest "I don't know" responses** are core requirements, not nice-to-haves.

**The knowledge base** is in [knowledge-base/](knowledge-base/) and contains 5 subfolders:
- `products/` — specs and prices for each device
- `faqs/` — common customer questions
- `policies/` — formal return, warranty, and privacy policies
- `support_tickets/` — past resolved customer issues
- `company/` — general info about TechNova

---
## 3. Tech Stack You're Expected to Use

You may choose your own tools, but here is the **recommended starting point** based on what was taught in Week 5:

| Component | Recommended | Alternative |
|---|---|---|
| Document loading | LangChain `DirectoryLoader` or a custom Python loop | `pathlib.Path.rglob` |
| Text splitting | LangChain `RecursiveCharacterTextSplitter` | Custom splitter |
| Embeddings | HuggingFace `all-MiniLM-L6-v2` (free, local) | OpenAI `text-embedding-3-small` (paid) |
| Vector store | Chroma (with persistence) | FAISS |
| LLM | `gpt-4.1-nano` (cheap, fast) | Ollama with `llama3.2` (free, local) |
| UI | Gradio `ChatInterface` | Streamlit |
| Eval | Python script with optional LLM-as-judge | Manual scoring |

**Pick one stack and stick with it.** Don't waste 2 days swapping tools.

---
## 4. Step-by-Step Requirements

The project is broken into 6 stages. Complete them in order. Test each stage before moving on.

### Stage 1 — Document Ingestion

**Goal:** Load every `.md` file from `knowledge-base/` into Python objects you can work with.

**Requirements:**
- Walk through all 5 subfolders.
- For each file, capture:
  - The full text content
  - The `source` (the file path, relative or absolute)
  - The `category` (the subfolder name: `products`, `faqs`, `policies`, `support_tickets`, or `company`)

**Think about:**
- Why is metadata important? (Hint: filtering, citations, debugging.)
- What happens if you forget to track which folder a file came from?
- Is there a difference between using a recursive folder walker vs hardcoding folder names? Which is more robust?

**Sanity check before moving on:**
- How many documents did you load? (Should be around 17.)
- Print the metadata of 3 random documents. Does it look right?

### Stage 2 — Chunking

**Goal:** Split each document into smaller, overlapping chunks suitable for embedding and retrieval.

**Requirements:**
- Choose a chunk size (typically **500–800 characters**) and overlap (typically **100–150 characters**).
- Every chunk must carry forward the metadata from its parent document (`source` and `category`).
- All chunks should fit comfortably inside the LLM's context window.

**Think about:**
- **Why overlap?** What happens at a chunk boundary if there's no overlap?
- **Why not just embed the whole document?** What's wrong with chunks that are too big?
- **What's wrong with chunks that are too small?**
- Markdown files have natural structure (headers, lists). Should your splitter respect that?

**Common mistakes:**
- Losing metadata after splitting (chunks have no `source` anymore).
- Splitting in the middle of a sentence and breaking meaning.
- Making chunks so large that retrieval becomes useless because every chunk "sort of" matches.

**Sanity check:**
- How many chunks did you produce in total?
- Print 2 random chunks. Do they have metadata? Do they read like coherent text?

### Stage 3 — Embeddings & Vector Store

**Goal:** Turn each chunk into a vector and store it in a persisted vector database.

**Requirements:**
- Use a single embedding model consistently (don't mix HuggingFace and OpenAI in one DB).
- Persist the vector store to a folder (e.g., `vector_db/`) so it survives kernel restarts.
- Add the chunks **with their metadata** — not just the text.
- Add some logic so the DB doesn't get rebuilt every single run (check if it already exists).

**Think about:**
- Why do we need a vector store at all? Why not just store the text and search with keywords?
- What does "semantic similarity" mean in this context?
- If you change your embedding model halfway through the project, what must you redo?
- Roughly how long will it take to embed ~100 chunks? (Hint: HuggingFace local is slow on first load; OpenAI API is fast but costs money.)

**Sanity check:**
- After creation, query the collection size — does it match your chunk count?
- Pull one record back and inspect its metadata.
- Restart your kernel and confirm the DB still loads without re-embedding.

### Stage 4 — Retrieval with Metadata Filtering

**Goal:** Given a user question, return the top-k most relevant chunks. Support filtering by category.

**Requirements:**
- Implement a function that takes a `question: str` and returns the top-k chunks (default k=5).
- Implement a way to pass in an optional `category` filter (e.g., `category="policies"` to only search policy docs).
- Each returned chunk must carry its metadata back to the caller so you can cite sources later.

**Think about:**
- What value of `k` works best? Try k=3, k=5, k=10. Look at the actual chunks returned.
- When would a user want to filter by category? (Hint: a support rep handling a returns inquiry probably only cares about `policies` and `faqs`.)
- What's the syntax for filtering in Chroma? (Look it up — this is part of the learning.)

**Try these queries manually before moving on:**
- "What is the return window for NovaCare+ members?" — should retrieve from `policies/`
- "How much is the Nova Phone X1?" — should retrieve from `products/`
- "Tell me about the cracked watch ticket" — should retrieve from `support_tickets/`
- "purple banana spaceship" — should retrieve... something? Inspect the score.

**If retrieval is bad, do NOT proceed to generation.** A broken retriever cannot be saved by a good LLM.

### Stage 5 — Generation with Source Citations

**Goal:** Combine the retrieved chunks with the user question and use an LLM to produce a grounded, cited answer.

**Requirements:**
- Build a system prompt that:
  - Tells the LLM it is an assistant for TechNova's support team.
  - Provides the retrieved chunks as context, each labeled with its source filename.
  - Instructs the LLM to cite the sources it uses in its answer.
  - Instructs the LLM to say "I don't know" or "I don't have that information" when the context is irrelevant.
- Pass the user's message and the conversation history.
- Return a response that includes inline source references (e.g., `(source: policies/return_policy.md)`).

**Think about:**
- How do you stop the LLM from making up sources? (Hint: only put sources in the prompt that you actually retrieved.)
- How do you stop the LLM from answering with its general knowledge instead of the context? (Hint: explicit instructions help, but they're not bulletproof.)
- What temperature should you use for a support assistant? Why?
- What happens if the retrieved chunks contradict each other?

**Test your generation with these queries:**
1. "What's the return window?" — should give a clear answer with citations.
2. "How do I make pasta?" — should refuse honestly.
3. "Tell me everything about the Nova Phone X1" — should pull multiple chunks and synthesize.

### Stage 6 — Gradio Chat UI

**Goal:** Wrap everything in a user-friendly chat interface.

**Requirements:**
- Use `gr.ChatInterface` (or build a custom chat with `gr.Blocks`).
- Multi-turn conversation — the bot should remember previous messages in the session.
- Show the user the citations clearly (either inline in the answer or in a separate area).
- (Optional) Add a dropdown for the user to pick a category filter (e.g., "All / Products only / Policies only").

**Think about:**
- The retrieval is usually based on the **latest message**. But what if the user says "What about for the 46mm one?" — that requires conversation context. How would you handle this? (Hint: this is what query rewriting is for — a stretch goal.)
- A support rep might want to see exactly which chunks were retrieved. Should you expose that for debugging?

**Sanity check:**
- Launch your app. Ask it 3 questions. Does it remember the conversation?
- Restart and ask a tricky one — does it cite sources correctly?

---
## 5. Evaluation Harness (Required)

**This is a separate script — `evaluate.py` — not part of your main solution notebook.**

### What it must do

1. **Load** the evaluation set from `evaluation/questions.json`.
2. **Loop** through every question.
3. For each question:
   - Send it to your RAG pipeline.
   - Capture the generated answer.
   - Capture the retrieved sources.
   - Score it.
4. **Aggregate** results into an overall score and save to `eval_results.json`.

### How to score

You may choose **one** of these methods (or combine them — bonus points for that):

**Method A — Keyword/Substring Match (simple, deterministic)**
- For each question, the eval JSON includes a `must_mention` list of keywords.
- Score = fraction of those keywords that appear in your generated answer.
- Pros: Cheap, fast, reproducible.
- Cons: Brittle — "5,000 mAh" vs "5000mAh" might not match unless you normalize.

**Method B — LLM-as-Judge (smart, expensive)**
- Pass the question, the expected facts, and your generated answer to an LLM.
- Ask it to score on a 0–5 scale and explain why.
- Pros: Catches paraphrased correct answers.
- Cons: Costs money, slightly non-deterministic.

### Additionally, score retrieval

For each question, check whether **at least one** of the `expected_sources` appears in your retrieved chunks. This is your **retrieval precision** metric.

### Special question type

Note that one question (`q16`) is **out-of-scope** on purpose. Your assistant should **NOT** invent an answer. Your evaluation should reward refusals on this question.

### Output format

Your `eval_results.json` should look something like this (you can extend it):

```json
{
  "summary": {
    "total_questions": 16,
    "retrieval_precision": 0.875,
    "answer_score_avg": 0.82,
    "refused_correctly": true
  },
  "per_question": [
    {
      "id": "q01",
      "question": "...",
      "predicted_answer": "...",
      "retrieved_sources": ["products/nova_phone_x1.md"],
      "score": 1.0
    }
  ]
}
```

---
## 6. Definition of Done

Before you submit, your project must satisfy **ALL** of these checks. Print this list and tick each one off.

- [ ] All 17 markdown files from `knowledge-base/` are ingested
- [ ] Every chunk has both `source` and `category` metadata
- [ ] Vector store is persisted to disk and reloads without re-embedding
- [ ] Retrieval returns sensible chunks for at least 5 manual test queries
- [ ] Category filter works (verify with a Chroma `where=` query)
- [ ] Generated answers include inline source citations
- [ ] Out-of-scope questions are refused, not hallucinated
- [ ] Gradio UI launches and supports multi-turn chat
- [ ] `evaluate.py` runs end-to-end and produces `eval_results.json`
- [ ] You scored at least **60%** on the evaluation set (anything less, debug your retrieval first)
- [ ] `REPORT.md` is written and explains your design choices
- [ ] Notebook runs cleanly on a fresh kernel (Restart & Run All passes)
- [ ] `.env` is NOT in your submission, but `.env.example` IS

---
## 7. Where to Look When You Get Stuck

**Re-read these in order:**

1. [HINTS.ipynb](HINTS.ipynb) — common pitfalls, sanity checks, debugging tips.
2. Week 5 notebook — the basic LangChain + Chroma RAG. This is closest to what you're building.
3. Week 5 notebook (PRO) — for ideas on chunking, reranking, query rewriting.
4. Official docs:
   - LangChain: https://python.langchain.com/docs/
   - Chroma: https://docs.trychroma.com/
   - Gradio: https://www.gradio.app/docs/

**Then** post on the course channel with a clear question — not "my code doesn't work" but "I'm getting X behavior, I expected Y, here's what I tried."

---
## 8. A Final Note

RAG looks deceptively simple. Anyone can wire up a retriever and a chat model in 30 lines of code and feel like they've built something.

The **real work** is in the details that this project pushes you toward:

- Metadata that travels with chunks all the way to the citation.
- Honest refusals when the knowledge base doesn't have the answer.
- An evaluation harness that tells you whether your changes are actually helping.
- A user interface that gives the human the trust signals they need.

If you build this well, you have built a **real product**, not a toy. Take that seriously.

Good luck — and don't forget to test on a fresh kernel before submitting!